In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 25.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 108.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.2 MB/s eta 0:0

In [2]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer , SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
model,tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit=True,
    dtype=None
)

==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.6.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [13]:
cpt_dataset = load_dataset("cyberpsych/PubMed-Cancer-NLP-Textual-Dataset", split="train")

In [15]:
def format_and_append_eos(example):
    title = str(example.get("Title", example.get("title", "")))
    abst = str(example.get("Abstract", example.get("abstract", example.get("text", ""))))

    combined_text = f"Title: {title}\nAbstract: {abst}\n"
    
    return {"text": combined_text + tokenizer.eos_token}

In [16]:
original_columns = cpt_dataset.column_names
cpt_dataset = cpt_dataset.map(format_and_append_eos, remove_columns=original_columns)

Map:   0%|          | 0/99956 [00:00<?, ? examples/s]

In [17]:
args = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = True,
    dataset_num_proc = 3,
)

In [18]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = cpt_dataset,
    args = args,
)

Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/99956 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=3):   0%|          | 0/99956 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [19]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,012 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.143665
2,2.131549
3,1.923315
4,1.973334
5,1.944726
6,1.879425
7,1.856117
8,2.019389
9,1.845674
10,1.808872


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [20]:
adapter_name = "qwen2.5-3b-oncology-lora"
model.save_pretrained(adapter_name)
tokenizer.save_pretrained(adapter_name)

Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-oncology-lora/tokenizer_config.json.


('qwen2.5-3b-oncology-lora/tokenizer_config.json',
 'qwen2.5-3b-oncology-lora/tokenizer.json')

In [21]:
!zip -r qwen2.5-3b-oncology-lora.zip /kaggle/working/qwen2.5-3b-oncology-lora

  adding: kaggle/working/qwen2.5-3b-oncology-lora/ (stored 0%)
  adding: kaggle/working/qwen2.5-3b-oncology-lora/tokenizer.json (deflated 81%)
  adding: kaggle/working/qwen2.5-3b-oncology-lora/tokenizer_config.json (deflated 89%)
  adding: kaggle/working/qwen2.5-3b-oncology-lora/README.md (deflated 65%)
  adding: kaggle/working/qwen2.5-3b-oncology-lora/adapter_config.json (deflated 57%)
  adding: kaggle/working/qwen2.5-3b-oncology-lora/adapter_model.safetensors (deflated 8%)


In [22]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [29]:
test_prompt = "Title: Efficacy of Immune Checkpoint Inhibitors in Metastatic Melanoma\nAbstract:"
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

In [32]:
with model.disable_adapter():
    base_outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
    print(tokenizer.batch_decode(base_outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Title: Efficacy of Immune Checkpoint Inhibitors in Metastatic Melanoma
Abstract: The incidence of melanoma is increasing in the United States, and the majority of patients with metastatic melanoma will eventually develop resistance to the current standard of care, which is a combination of immunotherapy and chemotherapy. The development of new therapies is urgently needed to improve the survival of patients with metastatic melanoma. Immune checkpoint inhibitors (ICIs) are a new class of drugs that have shown promise in the treatment of metastatic melanoma. In this review, we discuss the efficacy of ICIs in the treatment of metastatic melanoma, including the use of ICIs in combination with other therapies, the role of biomarkers in predicting response to ICIs, and the potential for combination therapy with ICIs and targeted agents. We also discuss the challenges and limitations of using ICIs in the treatment of metastatic melanoma, including the need for more effective biomarkers and th

In [33]:
ft_outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
print(tokenizer.batch_decode(ft_outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Title: Efficacy of Immune Checkpoint Inhibitors in Metastatic Melanoma
Abstract: Immune checkpoint inhibitors (ICIs) have revolutionized the treatment of metastatic melanoma, offering durable responses and improved survival rates. This review summarizes the current understanding of the mechanisms of action and efficacy of ICIs in melanoma, highlighting the role of immune checkpoint pathways, the clinical benefits observed with ICI therapy, and the ongoing challenges in optimizing treatment strategies. The review also discusses the potential for combination therapies and the role of biomarkers in predicting response to ICI therapy. The future of ICI therapy in melanoma is promising, with ongoing research aimed at improving treatment outcomes and identifying new targets for therapeutic intervention.

